# 🎨 Entrenamiento de Difusor con LoRA — Flickr8k

**Antes de correr:** Ve a `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`

In [ ]:
# ─── Celda 1: Verificar GPU ───────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─── Celda 2: Instalar dependencias ──────────────────────────────────────────
!pip install -q diffusers transformers peft datasets accelerate

In [ ]:
# ─── Celda 3: Montar Google Drive (para guardar el modelo) ───────────────────
from google.colab import drive
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/lora_model"
print(f"El modelo se guardará en: {output_dir}")

In [ ]:
# ─── Celda 4: Configuración ───────────────────────────────────────────────────
import os
import random
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from tqdm.notebook import tqdm

device     = "cuda"
model_id   = "runwayml/stable-diffusion-v1-5"
epochs     = 3
lr         = 1e-4
batch_size = 4
gradient_accumulation_steps = 4
max_samples = 5000   # None = todo el dataset (~40k filas)
image_size  = 512

print("Configuración lista")

In [ ]:
# ─── Celda 5: Cargar modelo base ─────────────────────────────────────────────
print("Cargando modelo base (puede tardar ~2 min)...")

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["to_q", "to_v"],
    lora_dropout=0.1,
    bias="none",
)
pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

pipe.unet.to(device)
pipe.vae.to(device)
pipe.text_encoder.to(device)

print("Modelo listo")

In [ ]:
# ─── Celda 6: Cargar dataset Flickr8k ────────────────────────────────────────
class Flickr8kDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, transform, caption_cols):
        self.data         = hf_dataset
        self.tokenizer    = tokenizer
        self.transform    = transform
        self.caption_cols = caption_cols

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample  = self.data[idx]
        image   = sample["image"].convert("RGB")
        image   = self.transform(image)
        col     = random.choice(self.caption_cols)
        caption = sample[col]
        tokens  = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt",
        )
        return {
            "pixel_values": image,
            "input_ids":    tokens.input_ids.squeeze(0),
            "attention_mask": tokens.attention_mask.squeeze(0),
        }

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

print("Descargando dataset Flickr8k...")
raw_dataset = load_dataset("jxie/flickr8k", split="train")
if max_samples:
    raw_dataset = raw_dataset.select(range(min(max_samples, len(raw_dataset))))

sample_keys  = list(raw_dataset[0].keys())
caption_cols = [c for c in sample_keys if c.startswith("caption")]
print(f"Muestras: {len(raw_dataset)} | Captions por imagen: {caption_cols}")

dataset    = Flickr8kDataset(raw_dataset, pipe.tokenizer, transform, caption_cols)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=True)
print("Dataset listo")

In [ ]:
# ─── Celda 7: Entrenamiento ───────────────────────────────────────────────────
optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=lr, weight_decay=1e-2)
scaler    = torch.amp.GradScaler("cuda")

pipe.unet.train()

for epoch in range(epochs):
    print(f"\n── Epoch {epoch+1}/{epochs} ──")
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(dataloader)):
        pixel_values   = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            encoder_hidden_states = pipe.text_encoder(
                input_ids, attention_mask=attention_mask
            ).last_hidden_state

        noise      = torch.randn_like(latents)
        timesteps  = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],), device=device
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        with torch.amp.autocast("cuda"):
            noise_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float())

        scaler.scale(loss / gradient_accumulation_steps).backward()

        if (step + 1) % gradient_accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Loss promedio: {total_loss / len(dataloader):.4f}")

print("\n✅ Entrenamiento terminado")

In [ ]:
# ─── Celda 8: Guardar modelo en Google Drive ──────────────────────────────────
os.makedirs(output_dir, exist_ok=True)
pipe.unet.save_pretrained(output_dir)
print(f"Modelo guardado en: {output_dir}")

In [ ]:
# ─── Celda 9: Prueba rápida de generación ─────────────────────────────────────
from IPython.display import display

pipe.unet.eval()
pipe.unet.to(device)

prompt = "a dog running on a beach at sunset, photorealistic"  # cambia esto

with torch.no_grad():
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]

display(image)
image.save("/content/prueba.png")
print(f"Prompt: {prompt}")